<a href="https://colab.research.google.com/gist/nmquintero/0a9789d29ef9f3b2089929e398e838d4/optimization-in-ai-practical-task.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Optimización en IA — Práctica

Este notebook implementa **3 algoritmos de optimización** con código comentado en español.

| Task | Algoritmo | Concepto |
|------|-----------|----------|
| 1 | Greedy | Decisiones locales rápidas |
| 2 | Brute Force | Enumeración exhaustiva |
| 3 | Programación Lineal | Modelo matemático + Solver |


In [1]:
# ─────────────────────────────────────────────────────────────
# CELDA DE CONFIGURACIÓN
# Instala las librerías necesarias (solo si falta alguna)
# ─────────────────────────────────────────────────────────────
import sys
import subprocess

def install_package(package: str) -> None:
    """Instala un paquete si no está disponible."""
    try:
        __import__(package)
        print(f"  ✅ {package} ya está instalado")
    except ImportError:
        print(f"  📦 Instalando {package}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", package])
        print(f"  ✅ {package} instalado")

install_package("pulp")
print("\n  🚀 Todo listo para comenzar")

  ✅ pulp ya está instalado

  🚀 Todo listo para comenzar


---
# 🧠 **Task 1 — Resource Scheduling (Greedy)**

### 📌 Problema
Asignar tareas con duración y recursos necesarios a recursos de capacidad limitada, cumpliendo un deadline.

### 💡 Algoritmo Greedy
Toma decisiones localmente óptimas: **ordena las tareas de menor a mayor duración** y las va asignando si hay recurso disponible. No garantiza el óptimo global, pero es rápido.

### 🧩 Pasos
1. Ordenar tareas por duración (más cortas primero)
2. Para cada tarea: buscar recurso con capacidad suficiente
3. Si cumple y no excede el deadline → asignar
4. Actualizar tiempo del recurso

### Complejidad: O(N × M)  (N = tareas, M = recursos)

In [2]:
# ─────────────────────────────────────────────────────────────
# TASK 1 — Resource Scheduling (GREEDY)
# ─────────────────────────────────────────────────────────────

# ─── DATOS DE ENTRADA ──────────────────────────────────────
# Cada tarea: (nombre, duracion_en_horas, [recursos_necesarios])
tareas = [
    ("Tarea A", 3, ["CPU", "RAM"]),    # Necesita CPU y RAM
    ("Tarea B", 1, ["CPU"]),           # Solo necesita CPU
    ("Tarea C", 5, ["GPU", "RAM"]),    # Necesita GPU y RAM
    ("Tarea D", 2, ["CPU", "GPU"]),    # Necesita CPU y GPU
    ("Tarea E", 4, ["RAM"]),           # Solo necesita RAM
    ("Tarea F", 2, ["CPU", "RAM"]),    # Necesita CPU y RAM
]

# Cada recurso: {"nombre": str, "capacidad": int (unidades de trabajo)}
recursos = [
    {"nombre": "CPU", "capacidad": 6},
    {"nombre": "GPU", "capacidad": 4},
    {"nombre": "RAM", "capacidad": 5},
]

# Deadline global (horas) — todas las tareas deben terminar antes
deadline = 15


# ─── ALGORITMO GREEDY ─────────────────────────────────────
def resource_scheduling_greedy(tareas, recursos, deadline):
    """
    Asigna tareas a recursos usando greedy (shortest-task-first).

    Pasos:
      1. Ordenar tareas por duración (↗ ascendente)
         → Las más cortas primero porque "encajan" más fácil
      2. Inicializar tiempo_acumulado de cada recurso en 0
      3. Para cada tarea (de la más corta a la más larga):
         a. Buscar recurso que: (i) sea necesario, (ii) tenga capacidad,
            (iii) no exceda el deadline
         b. Si encuentra → asigna y actualiza tiempo del recurso
         c. Si no → la tarea queda sin asignar
      4. Retornar plan + tiempo total
    """
    # 1. Ordenamos tareas por duración (sorted con key=lambda t: t[1])
    tareas_ordenadas = sorted(tareas, key=lambda t: t[1])

    # 2. Tiempo ocupado de cada recurso (empiezan en 0, todos disponibles)
    tiempo_recurso = {r["nombre"]: 0 for r in recursos}

    # 3. Plan de asignación: {nombre_tarea: (recurso, inicio, fin)}
    plan = {}
    no_asignadas = []

    for nombre_tarea, duracion, recursos_necesarios in tareas_ordenadas:
        asignada = False

        # Recorremos los recursos en orden
        for recurso in recursos:
            nom = recurso["nombre"]
            cap = recurso["capacidad"]

            # ⛔ Verificación 1: este recurso es de los que necesita la tarea?
            if nom not in recursos_necesarios:
                continue

            # ⛔ Verificación 2: hay capacidad disponible?
            if tiempo_recurso[nom] + duracion > cap:
                continue

            # ⛔ Verificación 3: entra dentro del deadline?
            if tiempo_recurso[nom] + duracion > deadline:
                continue

            # ✅ Pasa todas las verificaciones → ASIGNAMOS
            inicio = tiempo_recurso[nom]
            fin = inicio + duracion
            plan[nombre_tarea] = (nom, inicio, fin)
            tiempo_recurso[nom] = fin
            asignada = True
            break  # Salimos del loop de recursos, siguiente tarea

        if not asignada:
            no_asignadas.append(nombre_tarea)

    # 4. Tiempo total = suma de duraciones de tareas asignadas
    tiempo_total = sum(t[1] for t in tareas if t[0] in plan)

    return plan, tiempo_total, no_asignadas


# ─── EJECUCIÓN ────────────────────────────────────────────
plan, tiempo_total, no_asignadas = resource_scheduling_greedy(tareas, recursos, deadline)

# ─── RESULTADOS ───────────────────────────────────────────
print("╔════════════════════════════════════════════╗")
print("║   PLAN DE ASIGNACIÓN (Greedy)              ║")
print("╚════════════════════════════════════════════╝\n")

for nombre, duracion, _ in sorted(tareas, key=lambda t: t[1]):
    if nombre in plan:
        recurso, inicio, fin = plan[nombre]
        print(f"  {nombre:10s} → {recurso:5s}   [{inicio:2d} - {fin:2d}]  ✅")
    else:
        print(f"  {nombre:10s} →     —                ❌")

print(f"\n  ⏱  Tiempo total: {tiempo_total}h  |  Deadline: {deadline}h")
print(f"  📊 Asignadas: {len(plan)}/{len(tareas)}")
if no_asignadas:
    print(f"  ⚠️  No asignadas: {', '.join(no_asignadas)}")

print("\n" + "─" * 50)
print("📘 COMPLEJIDAD: O(N × M)  —  N = tareas, M = recursos")
print("⚠️  Greedy NO garantiza el óptimo global, solo una solución")
print("   rápida y 'suficientemente buena'.")

╔════════════════════════════════════════════╗
║   PLAN DE ASIGNACIÓN (Greedy)              ║
╚════════════════════════════════════════════╝

  Tarea B    → CPU     [ 0 -  1]  ✅
  Tarea D    → CPU     [ 1 -  3]  ✅
  Tarea F    → CPU     [ 3 -  5]  ✅
  Tarea A    → RAM     [ 0 -  3]  ✅
  Tarea E    →     —                ❌
  Tarea C    →     —                ❌

  ⏱  Tiempo total: 8h  |  Deadline: 15h
  📊 Asignadas: 4/6
  ⚠️  No asignadas: Tarea E, Tarea C

──────────────────────────────────────────────────
📘 COMPLEJIDAD: O(N × M)  —  N = tareas, M = recursos
⚠️  Greedy NO garantiza el óptimo global, solo una solución
   rápida y 'suficientemente buena'.


---
# 🚚 **Task 2 — Vehicle Routing Problem (Brute Force)**

### 📌 Problema
Un depot, varios clientes y vehículos con límite de paradas. Encontrar las rutas que minimicen la distancia total.

### 💡 Algoritmo Brute Force
**Enumera TODAS las permutaciones** posibles de los clientes, calcula la distancia de cada una y se queda con la mejor. Garantiza el óptimo global pero es O(N!).

### 🧩 Pasos
1. Generar todas las permutaciones de clientes (`itertools.permutations`)
2. Para cada permutación: dividir en K segmentos (uno por vehículo)
3. Calcular distancia: Depot → c1 → c2 → ... → Depot
4. Sumar distancias de todos los vehículos
5. Guardar la solución de menor distancia total

### Complejidad: O(N!) — solo factible para N ≤ 10

In [3]:
# ─────────────────────────────────────────────────────────────
# TASK 2 — Vehicle Routing Problem (BRUTE FORCE)
# ─────────────────────────────────────────────────────────────
import itertools
import math
import time

# ─── DATOS DE ENTRADA ──────────────────────────────────────
# El depot es el punto de partida y llegada de cada vehículo
depot = (0, 0)  # (x, y)

# Lista de clientes — cada uno es una coordenada (x, y)
clientes = [
    (2, 3),   # Cliente 1
    (5, 1),   # Cliente 2
    (1, 5),   # Cliente 3
    (6, 4),   # Cliente 4
    (3, 2),   # Cliente 5
]

# Número de vehículos y capacidad (máx clientes por ruta)
num_vehiculos = 2
capacidad_vehiculo = 3  # cada vehículo puede visitar hasta 3 clientes

N = len(clientes)


# ─── FUNCIONES AUXILIARES ─────────────────────────────────
def distancia(p1, p2):
    """Distancia euclidiana entre dos puntos."""
    return math.sqrt((p2[0] - p1[0])**2 + (p2[1] - p1[1])**2)


def distancia_ruta(ruta, todos_clientes):
    """
    Calcula la distancia total de una ruta que:
      - Sale del depot
      - Visita los clientes en orden
      - Vuelve al depot
    """
    total = 0.0
    total += distancia(depot, todos_clientes[ruta[0]])       # Depot → primer cliente
    for i in range(1, len(ruta)):
        total += distancia(todos_clientes[ruta[i-1]], todos_clientes[ruta[i]])  # entre clientes
    total += distancia(todos_clientes[ruta[-1]], depot)    # Último cliente → Depot
    return total


# ─── ALGORITMO BRUTE FORCE ───────────────────────────────
def vrp_bruteforce(clientes, num_vehiculos, capacidad_vehiculo):
    """
    Resuelve el VRP por fuerza bruta.

    1. Genera TODAS las permutaciones de clientes (N! posibilidades)
    2. Divide cada permutación en K segmentos (uno por vehículo)
    3. Calcula distancia total de cada solución
    4. Retorna la de menor distancia
    """
    n = len(clientes)
    indices = list(range(n))
    mejor_distancia = float('inf')
    mejores_rutas = []
    total_permutaciones = 0

    inicio = time.time()

    for perm in itertools.permutations(indices):
        total_permutaciones += 1

        # Dividimos la permutación en segmentos (uno por vehículo)
        rutas = []
        idx = 0
        for v in range(num_vehiculos):
            fin = min(idx + capacidad_vehiculo, n)
            ruta = list(perm[idx:fin])
            if ruta:
                rutas.append(ruta)
            idx = fin

        # Calculamos distancia total
        dist_total = sum(distancia_ruta(r, clientes) for r in rutas)

        # Guardamos si es mejor
        if dist_total < mejor_distancia:
            mejor_distancia = dist_total
            mejores_rutas = rutas

    fin = time.time()
    return mejores_rutas, mejor_distancia, total_permutaciones, fin - inicio


# ─── EJECUCIÓN ────────────────────────────────────────────
print("╔════════════════════════════════════════════╗")
print("║   VRP — BRUTE FORCE                        ║")
print("╚════════════════════════════════════════════╝\n")

print(f"  📍 Depot: {depot}")
for i, c in enumerate(clientes):
    print(f"  👤 Cliente {i+1}: {c}")
print(f"  🚛 Vehículos: {num_vehiculos}, Capacidad c/u: {capacidad_vehiculo}")
print(f"\n  ⚠️  {N}! = {math.factorial(N):,} permutaciones\n")

rutas, dist_total, total_perm, tiempo_ej = vrp_bruteforce(
    clientes, num_vehiculos, capacidad_vehiculo
)

print(f"  ⏱  Tiempo: {tiempo_ej:.4f}s")
print(f"  🔄 Permutaciones evaluadas: {total_perm:,}\n")

print("── Rutas óptimas ──\n")
for i, ruta in enumerate(rutas):
    puntos = " → ".join(["Depot"] + [f"Cliente {idx+1}{clientes[idx]}" for idx in ruta] + ["Depot"])
    print(f"  🚛 Vehículo {i+1}:\n     {puntos}")
    print(f"     Distancia: {distancia_ruta(ruta, clientes):.2f}\n")

print(f"  📏 Distancia TOTAL: {dist_total:.2f}")

print("\n" + "─" * 50)
print("📘 COMPLEJIDAD: O(N!)  —  crece factorialmente")
print("⚠️  Solo viable para N ≤ 10")
print("💡 Para + clientes: algoritmos genéticos, simulated annealing, etc.")

╔════════════════════════════════════════════╗
║   VRP — BRUTE FORCE                        ║
╚════════════════════════════════════════════╝

  📍 Depot: (0, 0)
  👤 Cliente 1: (2, 3)
  👤 Cliente 2: (5, 1)
  👤 Cliente 3: (1, 5)
  👤 Cliente 4: (6, 4)
  👤 Cliente 5: (3, 2)
  🚛 Vehículos: 2, Capacidad c/u: 3

  ⚠️  5! = 120 permutaciones

  ⏱  Tiempo: 0.0004s
  🔄 Permutaciones evaluadas: 120

── Rutas óptimas ──

  🚛 Vehículo 1:
     Depot → Cliente 2(5, 1) → Cliente 4(6, 4) → Cliente 5(3, 2) → Depot
     Distancia: 15.47

  🚛 Vehículo 2:
     Depot → Cliente 1(2, 3) → Cliente 3(1, 5) → Depot
     Distancia: 10.94

  📏 Distancia TOTAL: 26.41

──────────────────────────────────────────────────
📘 COMPLEJIDAD: O(N!)  —  crece factorialmente
⚠️  Solo viable para N ≤ 10
💡 Para + clientes: algoritmos genéticos, simulated annealing, etc.


---
# 📦 **Task 3 — Inventory Management (Linear Programming)**

### 📌 Problema
Demanda conocida por períodos, costos de pedido y almacenamiento. Minimizar costo total sin faltantes y sin exceder capacidad.

### 💡 Programación Lineal
A diferencia de greedy (local) y brute force (enumerar), acá **modelamos matemáticamente** el problema y un solver encuentra el óptimo global.

### 🧩 Modelo Matemático
- **Variables**: x[t] = pedido en período t, I[t] = inventario al final de t
- **Objetivo**: Minimizar Σ (costo_ordenar·x[t] + costo_almacenar·I[t])
- **Restricciones**: balance I[t] = I[t-1] + x[t] - D[t], I[t] ≥ 0, I[t] ≤ capacidad

### Complejidad: O(n²) en la práctica (Simplex) — escala a miles de variables

In [4]:
# ─────────────────────────────────────────────────────────────
# TASK 3 — Inventory Management (PROGRAMACIÓN LINEAL)
# ─────────────────────────────────────────────────────────────
import pulp
from pulp import LpMinimize, LpVariable, lpSum, LpStatus, value

# ─── DATOS DE ENTRADA ──────────────────────────────────────
# Demanda estimada para 12 meses
demanda = [100, 120, 150, 130, 160, 200, 180, 170, 140, 110, 90, 80]
nombres_meses = ["Ene","Feb","Mar","Abr","May","Jun",
                 "Jul","Ago","Sep","Oct","Nov","Dic"]

costo_ordenar = 5.0      # $ por unidad pedida
costo_almacenar = 2.0    # $ por unidad almacenada por mes
capacidad_max = 300.0    # capacidad máxima del almacén (unidades)

T = len(demanda)  # 12 períodos


# ─── MODELO DE PROGRAMACIÓN LINEAL ────────────────────────
def inventory_lp(demanda, costo_ordenar, costo_almacenar, capacidad_max):
    """
    Construye y resuelve el modelo de optimización.

    PASO 1: Crear el problema (minimización)
    PASO 2: Definir variables de decisión
      - x[t]: cantidad a pedir en período t  (x >= 0, continua)
      - I[t]: inventario al final de t       (0 <= I <= capacidad_max)
    PASO 3: Función objetivo
      - Minimizar Σ(costo_ordenar * x[t] + costo_almacenar * I[t])
    PASO 4: Restricciones
      - Balance: I[t] = I[t-1] + x[t] - D[t]
      - I[0] = 0 (inventario inicial vacío)
    PASO 5: Resolver con CBC (algoritmo Simplex)
    PASO 6: Obtener resultados
    """

    # PASO 1: Crear el problema
    problema = pulp.LpProblem("Gestion_de_Inventario", LpMinimize)

    # PASO 2: Variables de decisión
    x = [LpVariable(f"x_{t}", lowBound=0, cat="Continuous") for t in range(T)]
    I = [LpVariable(f"I_{t}", lowBound=0, upBound=capacidad_max, cat="Continuous") for t in range(T)]

    # PASO 3: Función objetivo
    problema += lpSum([costo_ordenar * x[t] + costo_almacenar * I[t] for t in range(T)]), "Costo_Total"

    # PASO 4: Restricciones (balance de inventario)
    for t in range(T):
        if t == 0:
            problema += I[t] == x[t] - demanda[t], f"Balance_{t}"
        else:
            problema += I[t] == I[t-1] + x[t] - demanda[t], f"Balance_{t}"

    # PASO 5: Resolver
    problema.solve(pulp.PULP_CBC_CMD(msg=False))

    # PASO 6: Resultados
    costo_total = value(problema.objective)
    return problema, x, I, costo_total


# ─── EJECUCIÓN ────────────────────────────────────────────
problema, x, I, costo_total = inventory_lp(demanda, costo_ordenar, costo_almacenar, capacidad_max)

# ─── RESULTADOS ───────────────────────────────────────────
print("╔════════════════════════════════════════════╗")
print("║   PLAN ÓPTIMO DE INVENTARIO (LP)           ║")
print("╚════════════════════════════════════════════╝\n")

print(f"  Estado del solver: {LpStatus[problema.status]}")
print(f"  💰 Costo total óptimo: ${costo_total:,.2f}\n")

print("  ┌──────────┬─────────┬─────────┬────────────┐")
print("  │ Período  │ Demanda │ Pedido  │ Inventario │")
print("  ├──────────┼─────────┼─────────┼────────────┤")
for t in range(T):
    print(f"  │ {nombres_meses[t]:8s} │ {demanda[t]:6.0f}  │ {value(x[t]):6.1f} │ {value(I[t]):9.1f} │")
print("  └──────────┴─────────┴─────────┴────────────┘\n")

costo_pedidos = sum(value(x[t]) * costo_ordenar for t in range(T))
costo_almac = sum(value(I[t]) * costo_almacenar for t in range(T))
print(f"  📋 Desglose:")
print(f"      Pedidos:       ${costo_pedidos:,.2f}")
print(f"      Almacenamiento: ${costo_almac:,.2f}")
print(f"      ───────────────────────────────")
print(f"      TOTAL:          ${costo_total:,.2f}")

print("\n" + "─" * 50)
print("📘 COMPLEJIDAD: O(n²) en la práctica (Simplex)")
print("✅ Garantiza el óptimo global")
print("💡 A diferencia de greedy (local) y brute force (enumerar),")
print("   LP modela el problema y el solver lo resuelve de forma inteligente.")

╔════════════════════════════════════════════╗
║   PLAN ÓPTIMO DE INVENTARIO (LP)           ║
╚════════════════════════════════════════════╝

  Estado del solver: Optimal
  💰 Costo total óptimo: $8,150.00

  ┌──────────┬─────────┬─────────┬────────────┐
  │ Período  │ Demanda │ Pedido  │ Inventario │
  ├──────────┼─────────┼─────────┼────────────┤
  │ Ene      │    100  │  100.0 │       0.0 │
  │ Feb      │    120  │  120.0 │       0.0 │
  │ Mar      │    150  │  150.0 │       0.0 │
  │ Abr      │    130  │  130.0 │       0.0 │
  │ May      │    160  │  160.0 │       0.0 │
  │ Jun      │    200  │  200.0 │       0.0 │
  │ Jul      │    180  │  180.0 │       0.0 │
  │ Ago      │    170  │  170.0 │       0.0 │
  │ Sep      │    140  │  140.0 │       0.0 │
  │ Oct      │    110  │  110.0 │       0.0 │
  │ Nov      │     90  │   90.0 │       0.0 │
  │ Dic      │     80  │   80.0 │       0.0 │
  └──────────┴─────────┴─────────┴────────────┘

  📋 Desglose:
      Pedidos:       $8,150.00
    

---
# 🔥 Comparación de los 3 enfoques

| Aspecto | Greedy | Brute Force | Programación Lineal |
|---------|--------|-------------|---------------------|
| **Óptimo global?** | ❌ No | ✅ Sí | ✅ Sí |
| **Velocidad** | ⚡ Rápido | 🐢 Lento | ⚡ Rápido |
| **Escalabilidad** | 📈 Alta | 📉 Baja | 📈 Alta |
| **Estrategia** | Decisiones locales | Probar todo | Modelar + Resolver |
| **Complejidad** | O(N·M) | O(N!) | O(n²) |
| **Dif. implementación** | Baja | Baja | Media |

### 💡 ¿Cuándo usar cada uno?
- **Greedy**: Cuando necesitamos una respuesta rápida y la optimalidad no es crítica
- **Brute Force**: Cuando N es pequeño (≤10) y necesitamos el óptimo exacto
- **Programación Lineal**: Cuando el problema se puede modelar matemáticamente y necesitamos escalabilidad + optimalidad